<div dir="rtl" align="right">

# التقييسُ مقابلُ التطبيعَ

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُقارنُ تقييسَ Z-score معَ تطبيعِ Min-Max على متجهاتِ السماتِ EEG.

## ماذا يَعمَلُ هذا الدفترُ؟

يُعيدُ تشكيلَ بياناتِ EEG إلى صورةٍ ثنائيّةِ الأبعادِ، ويُطبّقُ StandardScaler و MinMaxScaler، ويَطبعُ الإحصاءاتِ الناتجة.

## المُخرجاتُ المُتوقّعةُ

- ثلاثةُ مدرّجاتٍ تُظهرُ التوزيعَ قبلَ وبعدَ كلِّ طريقةِ قياسٍ
- البياناتُ المُقيّسةُ لها مُتوسطٌ ~0 وانحرافٌ معياريٌّ ~1
- البياناتُ المُطبّعةُ مُحصورةٌ بينَ 0 و 1

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| scaler | StandardScaler, MinMaxScaler |

</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تطبيقُ التقييسِ والتطبيعِ

</div>


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X_2d = X.reshape(n_trials, n_channels * n_samples)

# NOTE: In real ML pipelines, fit the scaler on training data only.
# This demo shows the effect of standardization on the full dataset.
scaler = StandardScaler()
X_std = scaler.fit_transform(X_2d)

normalizer = MinMaxScaler()
X_norm = normalizer.fit_transform(X_2d)

print(f'Original shape: {X_2d.shape}')
print(f'Raw mean: {X_2d.mean():.4f}, std: {X_2d.std():.4f}')
print(f'Standardized mean: {X_std.mean():.4f}, std: {X_std.std():.4f}')
print(f'Normalized min: {X_norm.min():.4f}, max: {X_norm.max():.4f}')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- البياناتُ الخامُ لها نطاقٌ واسعٌ من القيمِ مُتمركزةٌ قُربَ الصفرِ
- البياناتُ المُقيّسةُ مُتمركزةٌ عندَ 0 بِتباينٍ وحدويٍّ
- البياناتُ المُطبّعةُ مَضغوطةٌ في [0, 1]

</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

raw_flat = X_2d[:, :1000].flatten()
std_flat = X_std[:, :1000].flatten()
norm_flat = X_norm[:, :1000].flatten()

fig = make_subplots(rows=3, cols=1, subplot_titles=(
    'Raw feature distribution (first 1000 features)',
    'Standardized distribution (Z-score)',
    'Normalized distribution (Min-Max, 0-1)'))

fig.add_trace(go.Histogram(x=raw_flat, nbinsx=100, marker_color='steelblue', opacity=0.7), row=1, col=1)
fig.add_trace(go.Histogram(x=std_flat, nbinsx=100, marker_color='orange', opacity=0.7), row=2, col=1)
fig.add_trace(go.Histogram(x=norm_flat, nbinsx=100, marker_color='green', opacity=0.7), row=3, col=1)

fig.update_xaxes(title_text='Value', row=3, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_yaxes(title_text='Count', row=3, col=1)
fig.update_layout(height=900, showlegend=False, title_text='Standardization vs Normalization')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- التقييسُ (Z-score) يُمركِزُ البياناتِ عندَ 0 بِانحرافٍ معياريٍّ 1، مُحافظاً على شكلِ التوزيعِ
- التطبيعُ (Min-Max) يُقيّسُ البياناتِ إلى [0, 1]، وهوَ حسّاسٌ للقيمِ الشاذةِ
- الاختيارُ يَعتمدُ على الخوارزميّةِ اللاحقةِ: العديدُ من نماذجِ التعلّمِ الآليِّ تُفضّلُ التقييسَ

</div>
